In [ ]:
function range(start: number, stop: number): number[] {
    return Array.from({ length: stop - start + 1 }, (_, index) => start + index);
}

In [ ]:
range(3, 5)

In [ ]:
import { CSP, solve, Assignment, Variable, Formula } from "./02-Backtracking-Constraint-Solver";

# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missinaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any moments at any shore there are more infidels than missionaries, then the missionaries have a problem, since the infidels have a diet that includes human flesh.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossing of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{B}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the 
  $i^{\textrm{th}}$ crossing.

## Auxiliary Functions

The following functions generate the *formulas* required by the constraint solver. This ensures that the logical expressions can be evaluated dynamically using the variable names.

The invariant is given by the following formula:
$$M = 0 \vee M = 3 \vee M = C$$

In [ ]:
function invariant(M: Variable, C: Variable, B: Variable): Formula {
    return `${M} == 0 || ${M} == 3 || ${M} == ${C}`;
}

In [ ]:
invariant("M0", "C0", "B0")

The formula for the transition has three components:
* The boat changes sides with every transition:
  $$B_\beta = 1 - B_\alpha$$
* If the boat goes from the west to the east, there are two conditions:
  + The number of people on the boat must be at least 1 and at most two.
    $$1 \leq (M_\alpha - M_\beta) + (C_\alpha - C_\beta) \leq 2$$
  + The number of both missionaries and infidels on the western side must not increase.
    $$M_\beta \leq M_\alpha \wedge C_\beta \leq C_\alpha$$
* If the boat goes from the east to the west, there are two conditions:
  + The number of people on the boat must be at least 1 and at most two.
    $$ 1 \leq (M_\beta - M_\alpha) + (C_\beta - C_\alpha) \;\wedge\; 
              (M_\beta - M_\alpha) + (C_\beta - C_\alpha)\leq 2 $$
  + The number of both missionaries and infidels on the western side must not decrease.
    $$M_\beta \geq M_\alpha \wedge C_\beta \geq C_\alpha$$    

In [ ]:
function transition(M𝛼: Variable, C𝛼: Variable, B𝛼: Variable, 
                    M𝛽: Variable, C𝛽: Variable, B𝛽: Variable): Formula 
{
    const westToEast = [`(1 <= (${M𝛼} - ${M𝛽}) + (${C𝛼} - ${C𝛽})`,
                        `(${M𝛼} - ${M𝛽}) + (${C𝛼} - ${C𝛽}) <= 2`,
                        `${M𝛽} <= ${M𝛼} && ${C𝛽} <= ${C𝛼})`];
    const eastToWest = [`(1 <= (${M𝛽} - ${M𝛼}) + (${C𝛽} - ${C𝛼})`, 
                        `(${M𝛽} - ${M𝛼}) + (${C𝛽} - ${C𝛼}) <= 2`, 
                        `${M𝛽} >= ${M𝛼} && ${C𝛽} >= ${C𝛼})`];
    
    return `(${B𝛽} == 1 - ${B𝛼}) && ((${B𝛼} == 1) ? ${westToEast.join(' && ')} 
                                                  : ${eastToWest.join(' && ')})`;
}

The function `missionariesCSP` creates a CSP that tries to solve the problem with `n` crossings.

In [ ]:
function missionariesCSP(n: number): CSP {
    const Vars    = range(0, n).flatMap(i => [`M${i}`, `C${i}`, `B${i}`]);
    const Values  = range(0, 3);
    const Constrs = ['M0 == 3 && C0 == 3 && B0 == 1',          // Start
                     `M${n} == 0 && C${n} == 0 && B${n} == 0`, // Goal
                     ...range(0, n-1).map(i => invariant( `M${i}`, `C${i}`, `B${i}`)),
                     ...range(0, n-1).map(i => transition(`M${i}`, `C${i}`, `B${i}`, 
                                                          `M${i+1}`, `C${i+1}`, `B${i+1}`))];
    return [Vars, Values, Constrs];
}

In [ ]:
missionariesCSP(3);

The function `findSolution` computes a solution to the problem of saving the infidels by iteratively increasing the number of crossings `n`.

In [ ]:
function findSolution(): [number, Assignment] {
    let n = 1;
    while (true) {
        console.log(`Checking n = ${n}...`);
        const csp = missionariesCSP(n);
        const result = solve(csp);
        if (result != null) { return [n, result]; }
        n += 2;
    }
}

Finding the solution takes about 0.4 seconds on my Mac M2 Max Studio.

In [ ]:
console.time("findSolution"); 
const result = findSolution(); 
console.timeEnd("findSolution");
result

## Visualizing the Result

In [ ]:
function showSolution(solution: Assignment, n: number) {
    for (let i = 0; i <= n; i++) {
        const M = Number(solution.get(`M${i}`));
        const C = Number(solution.get(`C${i}`));
        const B = Number(solution.get(`B${i}`));
        
        const leftSide = '😇'.repeat(M) + '🥷'.repeat(C);
        const rightSide = '😇'.repeat(3 - M) + '🥷'.repeat(3 - C);
        const gap = " ".repeat(28 - (M + C) * 2);
        
        console.log(`${leftSide}${gap}${rightSide}`);
        
        if (i < n) {
            const nextM = Number(solution.get(`M${i+1}`));
            const nextC = Number(solution.get(`C${i+1}`));
            
            if (B == 1) {
                const MB = M - nextM;
                const CB = C - nextC;
                console.log(' '.repeat(10) + `>>> ${'😇'.repeat(MB)}${'🥷'.repeat(CB)} >>>`);
            } else {
                const MB = nextM - M;
                const CB = nextC - C;
                console.log(' '.repeat(10) + `<<< ${'😇'.repeat(MB)}${'🥷'.repeat(CB)} <<<`);
            }
        }
    }
}

In [ ]:
const [n, solution] = result;
showSolution(solution, n);

In [ ]:
import * as tslab from "tslab";

function show_animated_solution(solution: Assignment, n: number) {
    if (!solution) {
        console.error("No solution provided.");
        return;
    }

    // Prepare the state data for the frontend JavaScript
    const states = [];
    for (let i = 0; i <= n; i++) {
        states.push({ M: Number(solution.get(`M${i}`)), C: Number(solution.get(`C${i}`)), B: Number(solution.get(`B${i}`)) });
    }

    // Generate a unique ID suffix to avoid element collisions if run multiple times
    const uid = Math.random().toString(36).substring(2, 9);

    const htmlContent = `
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-${uid}" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-${uid}" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-${uid}" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-${uid}" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of ${n}</span>
        </div>
        
        <svg width="600" height="250" style="background-color: #e0f7fa; border: 1px solid #b2ebf2; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="250" fill="#a5d6a7" />
            <rect x="450" y="0" width="150" height="250" fill="#a5d6a7" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Western Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Eastern Shore</text>

            <text id="west-m-${uid}" x="75" y="100" font-size="28" text-anchor="middle">😇😇😇</text>
            <text id="west-c-${uid}" x="75" y="150" font-size="28" text-anchor="middle">🥷🥷🥷</text>
            
            <text id="east-m-${uid}" x="525" y="100" font-size="28" text-anchor="middle"></text>
            <text id="east-c-${uid}" x="525" y="150" font-size="28" text-anchor="middle"></text>

            <g id="boat-${uid}" style="transition: transform 1.5s ease-in-out;">
                <rect x="160" y="160" width="80" height="40" rx="8" fill="#8d6e63" stroke="#5d4037" stroke-width="2"/>
                <text id="boat-p-${uid}" x="200" y="188" font-size="22" text-anchor="middle"></text>
            </g>
        </svg>
    </div>

    <script>
    (function() {
        const states = ${JSON.stringify(states)};
        let currentStep = 0;
        let isAnimating = false;
        
        const boat = document.getElementById('boat-${uid}');
        const westM = document.getElementById('west-m-${uid}');
        const westC = document.getElementById('west-c-${uid}');
        const eastM = document.getElementById('east-m-${uid}');
        const eastC = document.getElementById('east-c-${uid}');
        const boatP = document.getElementById('boat-p-${uid}');
        const stepLabel = document.getElementById('label-step-${uid}');
        
        // Web Audio API context for the motor sound
        let audioCtx;
        function playMotorSound() {
            if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            const osc = audioCtx.createOscillator();
            const filter = audioCtx.createBiquadFilter();
            const gain = audioCtx.createGain();
            
            // DIESEL ENGINE SIMULATION:
            // A sawtooth wave at ~12 Hz creates a rapid "clicking/knocking" 
            // sound that mimics the slow chug of a diesel engine.
            osc.type = 'sawtooth';
            osc.frequency.value = 12; 
            
            // Low-pass filter to muffle the harsh highs and keep it sounding bassy/heavy
            filter.type = 'lowpass';
            filter.frequency.value = 150;
            gain.gain.setValueAtTime(0, audioCtx.currentTime);
            // Using a slightly higher gain (0.6) because extreme low frequencies are perceived as quieter
            gain.gain.linearRampToValueAtTime(0.6, audioCtx.currentTime + 0.2); 
            
            osc.connect(filter);
            filter.connect(gain);
            gain.connect(audioCtx.destination);
            osc.start();
            
            return { osc, gain };
        }
        
        function stopMotorSound(sound) {
            if (sound) {
                sound.gain.gain.linearRampToValueAtTime(0, audioCtx.currentTime + 0.4); // Slower fade out for realism
                setTimeout(() => sound.osc.stop(), 400);
            }
        }

        function renderStaticState(step) {
            const s = states[step];
            westM.textContent = '😇'.repeat(s.M);
            westC.textContent = '🥷'.repeat(s.C);
            eastM.textContent = '😇'.repeat(3 - s.M);
            eastC.textContent = '🥷'.repeat(3 - s.C);
            boatP.textContent = '';
            
            boat.style.transform = s.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of ${n}';
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const mPass = Math.abs(s0.M - s1.M);
            const cPass = Math.abs(s0.C - s1.C);
            
            if (s0.B === 1) { 
                westM.textContent = '😇'.repeat(s0.M - mPass);
                westC.textContent = '🥷'.repeat(s0.C - cPass);
            } else { 
                eastM.textContent = '😇'.repeat((3 - s0.M) - mPass);
                eastC.textContent = '🥷'.repeat((3 - s0.C) - cPass);
            }
            boatP.textContent = '😇'.repeat(mPass) + '🥷'.repeat(cPass);
            
            const sound = playMotorSound();
            boat.style.transform = s1.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1500));
            
            stopMotorSound(sound);
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-${uid}').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });

        document.getElementById('btn-prev-${uid}').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });

        document.getElementById('btn-reset-${uid}').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    `;

    tslab.display.html(htmlContent);
}

In [ ]:
show_animated_solution(solution, n);